# <font color="orange">1\. 学習させる前にデータの中身を理解する</font>

必用なライブラリをインポートする

In [ ]:
# 警告メッセージを無視するためにwarningsモジュールを使用
import warnings, requests, zipfile, io

# 警告メッセージを無視するフィルターを設定
warnings.simplefilter('ignore')

# データ処理のためにpandasモジュールをインポート
import pandas as pd

S3にアクセスして機械学習に使うデータを引っ張ってくる

In [ ]:
# boto3を使ってAWSのサービスにアクセスするためのライブラリをインポート
import boto3

# pandasライブラリを使って、AWS S3上のCSVファイルを読み込む
# 事前に必要なデータをS3にアップロードをして、URIをメモしておいてください。
df = pd.read_csv('s3://自分のS3のurl')

「**df.shape**」を使ってデータが何行、何列あるのか調べる

In [ ]:
# DataFrameの行数と列数を確認するためにshape属性を使用
df.shape

「**df.colums**」を使ってカラム(項目)を調べる
「**target**」列は乳がんか乳がんじゃないかを書いてる列

In [ ]:
# DataFrameの列名を取得するためにcolumns属性を使用
df.columns

「**df.dtypes**」各カラムのデータ型を調べる

In [ ]:
# DataFrameの各列のデータ型を確認するためにdtypes属性を使用
df.dtypes

最初の列の統計を出す

In [ ]:
# 特定の列（'mean radius'）の統計的な要約を取得するためにdescribeメソッドを使用
df['mean radius'].describe()

「**df.describe**」で各カラムの統計を出す

In [ ]:
df.describe()

表形式だったら見づらいから「**matplotlib**」でグラフでデータを可視化する(外れ値の確認もする)

In [ ]:
# データの可視化のためにmatplotlib.pyplotモジュールをインポート
import matplotlib.pyplot as plt

# Jupyter Notebook上でプロットを表示するために%matplotlib inlineを使用
%matplotlib inline

# DataFrameの全体的なプロットを生成するためにplotメソッドを使用
df.plot()

↑ でやったコマンドは一気に全部のカラムを一緒のグラフに出力して見づらいから各カラムに分けてデータを出力する

In [ ]:
# 密度プロットを生成するためにplotメソッドを使用し、subplotsをTrueに設定
# layoutでプロットの配置を指定し、figsizeでプロットのサイズを設定
# sharexをFalseに設定してx軸を共有しないようにする
df.plot(kind='density', subplots=True, layout=(7, 5), figsize=(20, 20), sharex=False)

# プロットを表示
plt.show()

出力した結果「**mean area**」というカラムに外れ値がありそうだから**mean area**単体でグラフを出力する

In [ ]:
# 特定の列（'mean area'）に対して密度プロットを生成するためにplot.density()メソッドを使用
df['mean area'].plot.density()

出力した結果**2500**付近で数値が増加しているから**ヒストグラム**でより外れ値をより分かりやすく可視化する

In [ ]:
# 特定の列（'mean area'）に対してヒストグラムを生成するためにplot.histメソッドを使用
df['mean area'].plot.hist()

**ヒストグラム**よりも分かりやすくデータを可視化するために**箱ひげ図**も活用する

In [ ]:
# 特定の列（'mean area'）の箱ひげ図を生成するためにplotメソッドを使用
df['mean area'].plot.box()

模範解答が言うには2500付近の数値の外れ値は別に大きく外れてるわけじゃないからそのままで問題ないらしい<br>
****

target(答え)の健常者が何件か、異常者が何件かを調べる

In [ ]:
# 特定の列（'target'列）の各値の出現回数を取得するためにvalue_countsメソッドを使用
df['target'].value_counts()

乳がんのデータの場合は3/2が正常で3/1が異常だったらしい
***

正常者と異常者に分けてmeanareaの数値がどんな感じで分布されてるのか見る

In [ ]:
# 散布図を生成するためにplot.scatterメソッドを使用
# y軸に'mean area'列、x軸に'target'列を指定
df.plot.scatter(y='mean area', x='target')

AWSは**mean area**に限らずほかのカラムのデータの可視化をしている

In [ ]:
# 'target'列でグループ分けして、各グループのボックスプロットを生成
# fontsizeでフォントサイズを指定し、rotでx軸のラベルを回転させる角度を指定
# figsizeでプロットのサイズを指定し、patch_artistをTrueに設定してボックスの塗りつぶしを有効にする
df.groupby('target').boxplot(fontsize=20, rot=90, figsize=(20, 10), patch_artist=True)

とりあえずデータの相関とかプロップとで出力してる

In [ ]:
# 相関行列を計算し、DataFrame（df）から取得
corr_matrix = df.corr()

# "target"列に対する相関係数を降順で並べ替え
corr_matrix["target"].sort_values(ascending=False)

In [ ]:
# 散布図行列を生成するためにpandas.plotting.scatter_matrix関数を使用
# figsizeでプロットのサイズを設定
pd.plotting.scatter_matrix(df, figsize=(12, 12))

# プロットを表示
plt.show()

**sea born**でもデータの可視化をしてる

# <font color="orange">2\. データの準備からモデルの作成</font><br>
****

まずXGboostを使うから学習に使うデータのtarget列(答えの列)を末尾から先頭に持ってくる

In [ ]:
# DataFrameの列名をリストに変換
cols = df.columns.tolist()

# 先頭の列を末尾に移動するためにスライスを使用
cols = cols[-1:] + cols[:-1]

# 列の順序を変更するためにDataFrameを再構築
df = df[cols]

**target**列が先頭に来てたらOK

【元のデータ】<br>
Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
        ......
       'worst concave points', 'worst symmetry', 'worst fractal dimension',
       '<font color="red">target</font>'],
      dtype='object')


【並び変えたデータ】<br>
Index(['<font color="red">target</font>', 'mean radius', 'mean texture', 'mean perimeter', 'mean area',
        ......
       'worst concave points', 'worst symmetry', 'worst fractal dimension'],
      dtype='object')

In [ ]:
# DataFrameの列名を表示するためにcolumns属性を使用
df.columns

学習に使うデータを
- **train** (モデルに学習させる用のデータ)
- **validate** (学習のチューニングに使うデータ)
- **test** (文字通りモデルにテストさせる用のデータ)
の三分割にする

【詳細】
| セット | 用途 | 割合 |
|-----|-----|-----|
|train|モデルに学習させる|80%|
|validate|学習中の調整に使う|10%|
|test|最適な性能評価に使う|10%|

In [ ]:
# 機械学習モデルのトレーニングとテストのためにデータを分割するためにtrain_test_split関数をインポート
from sklearn.model_selection import train_test_split

# データセットをトレーニングセットとテスト＆検証セットに分割するためにtrain_test_split関数を使用
# test_sizeでテスト＆検証セットの割合を指定し、random_stateで乱数のシードを固定
# stratifyでクラスの分布を保持して分割
train, test_and_validate = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])

↑ ↑ 最初は **train** と **test_and_validae** (文字通りvalidateとtestを分割しない変数)  のに分割にする 
***

次にtest_and_validateからに分割してようやく３分割できる

In [ ]:
# train_test_split関数を使用してデータを分割
# test_and_validateデータをtestとvalidateに分割し、test_sizeで分割比率を指定
# random_stateで再現性を確保し、stratifyで層別サンプリングを行う（'class'列を考慮）
test, validate = train_test_split(test_and_validate, test_size=0.5, random_state=42, stratify=test_and_validate['target'])

指定した割合でデータが分割されてるのか調べる

In [ ]:
# 訓練データセットの行数と列数を表示
print(train.shape)

# テストデータセットの行数と列数を表示
print(test.shape)

# 検証データセットの行数と列数を表示
print(validate.shape)

三分割に分けたデータの**正常**と**異常**の分かれ方を見る

In [ ]:
# 'target'列の値の出現回数を表示するためにvalue_countsメソッドを使用
print(train['target'].value_counts())

# テストデータの'target'列の値の出現回数を表示
print(test['target'].value_counts())

# 検証データの'target'列の値の出現回数を表示
print(validate['target'].value_counts())

## 3三分割したデータをS3に保存する

保存の準備 ↓

In [ ]:
# S3バケットとプレフィックスの設定
bucket='machinelearning0655'
prefix='lab3'

# 訓練データ、テストデータ、検証データのファイル名設定
train_file='vertebral_train.csv'
test_file='vertebral_test.csv'
validate_file='vertebral_validate.csv'

# osモジュールを使用してファイルパスを構築
import os

# boto3セッションからS3リソースを取得
s3_resource = boto3.Session().resource('s3')

# S3にCSVファイルをアップロードするための関数を定義
def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False)
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

保存する ↓

In [ ]:
# train_fileというCSVファイルをS3にアップロードするためにupload_s3_csv関数を使用
upload_s3_csv(train_file, 'train', train)

# test_fileというCSVファイルをS3にアップロードするためにupload_s3_csv関数を使用
upload_s3_csv(test_file, 'test', test)

# validate_fileというCSVファイルをS3にアップロードするためにupload_s3_csv関数を使用
upload_s3_csv(validate_file, 'validate', validate)

## ここからモデルのトレーニング
***

**コンテナ**(XGBoostが動く実行環境一式)のurlを取得する<br>
**xgboost**：使うアルゴリズム名<br>
**boto3.Session().region_name**：自分が使ってるリージョン名 ※<font color="red">リージョンがあってるかちゃんと確認すること!</font><br>
**1.0-1**：xgboostのバージョン

In [ ]:
# Amazon SageMaker用のモジュールをインポート
import boto3
from sagemaker.image_uris import retrieve

# SageMakerにおいてXGBoostアルゴリズムを利用するためのコンテナイメージを取得
container = retrieve('xgboost',boto3.Session().region_name,'1.0-1')


ハイパーパラメータの設定<br>
**ハイパーパラメータ**：人間側が事前に決める学習の仕方の設定

各設定の意味はこう。<br>
`num_round`: 42<br>
学習を何ラウンド繰り返すか。XGBoostは「弱いモデルを少しずつ改善していく」を繰り返す仕組みなので、何回繰り返すかを指定する。42回。<br>

`eval_metric`: auc<br>
「どの指標でモデルの良し悪しを測るか」。AUCは3_6で出てきた「モデルの判別能力の総合スコア」。1.0に近いほど優秀。<br>

`objective`: binary:logistic<br>
「何を解く問題か」を指定。binary=2択（癌か正常か）、logistic=結果を確率（0〜1）で返す、という意味。これがあるから3_5で「0.82」みたいな確率値が返ってきてた。

In [ ]:
# XGBoostのハイパーパラメータを設定するための辞書を作成
hyperparams={"num_round":"42",
            "eval_metric": "auc",
            "objective": "binary:logistic"}

**estimator**の設定<br>
**estimator**：「どのマシンで・どんな設定で学習させるか」をまとめた設計書みたいなもの

`container`：さっき取得したXGBoostの実行環境<br>

`get_execution_role()`：AWSの権限設定（S3にアクセスする許可など）<br>

`instance_count=1`：クラウドのマシンを何台使うか（1台）<br>

`instance_type='ml.m5.xlarge'`：マシンのスペック指定<br>

`output_path`：学習済みモデルの保存先S3パス<br>

`hyperparameters`：さっき設定したハイパーパラメータ

In [ ]:
# SageMaker SDKを使用して、XGBoostのモデルをトレーニングおよびデプロイするために必要なモジュールをインポート
import sagemaker

# 出力先のS3ロケーションを設定
s3_output_location="s3://{}/{}/output/".format(bucket,prefix)

# SageMakerのXGBoostアルゴリズムを使用するためにXGBoost Estimatorを作成
# container: 使用するコンテナの指定
# sagemaker.get_execution_role(): SageMakerの実行ロールを取得
# instance_count: モデルトレーニングに使用するインスタンスの数を指定
# instance_type: モデルトレーニングに使用するインスタンスのタイプを指定
# output_path: トレーニングアーティファクト（モデルアーティファクト）のS3出力先を指定
# hyperparameters: XGBoostのハイパーパラメータを指定
# sagemaker_session: SageMakerセッションを指定
xgb_model=sagemaker.estimator.Estimator(container,
                                    sagemaker.get_execution_role(),
                                    instance_count=1,
                                    instance_type='ml.m4.xlarge',
                                    output_path=s3_output_location,
                                        hyperparameters=hyperparams,
                                        sagemaker_session=sagemaker.Session())

データチャンネルの設定をする
アップロードしたデータ(さっき3分割したデータ)の保存場所をモデルに教える

In [ ]:
# SageMakerのトレーニングジョブに使用するトレーニングデータのS3パスとコンテントタイプを指定
train_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train/".format(bucket,prefix,train_file),
    content_type='text/csv')

# SageMakerのトレーニングジョブに使用する検証データのS3パスとコンテントタイプを指定
validate_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validate/".format(bucket,prefix,validate_file),
    content_type='text/csv')

# トレーニングジョブのデータチャネルにトレーニングデータと検証データを指定
data_channels = {'train': train_channel, 'validation': validate_channel}

**fit**でトレーニング開始!!<br>
※ ここのトレーニングで５～１０分かかる<br>
`Training job completed`って文字が出力されればトレーニングが完了

In [ ]:
# XGBoostモデルを学習させるためにfitメソッドを使用
# inputsにはデータチャネルを指定し、logsをFalseに設定してログを表示しないようにする
xgb_model.fit(inputs=data_channels, logs=False)

# ホスティングの準備が完了したことを表示
print('ready for hosting!')

# <font color="orange">3\. 完成させたモデル(作ったAI)に予測をさせてみる</font><br>
****

作ったモデルを「APIとして動くサーバー」に載せる

【今まで】
学習させたモデルをS3に保存してるだけ<br>
                ↓<br>
【これから】
HTTPでデータを送ると予測を返してくれるサーバーが立つようにする

In [ ]:
# XGBoostモデルをデプロイしてエンドポイントを作成
xgb_predictor = xgb_model.deploy(initial_instance_count=1, # インスタンスの数を指定
                serializer = sagemaker.serializers.CSVSerializer(), # シリアライザをCSV形式に設定
                instance_type='ml.m4.xlarge') # インスタンスタイプを指定
# ホスティングが完了したことを表示
print('Hosting completed!')

試しに1件だけデータを渡して予測させる



In [ ]:
# テストデータから1行目（0から始まる行）を取得し、最初の列以降のデータを表示
row = test.iloc[0:1,1:]

# CSV形式の文字列を一時的に格納するStringIOオブジェクトを作成
batch_X_csv_buffer = io.StringIO()

# DataFrameの1行をCSV形式でStringIOに書き込む（ヘッダーなし、インデックスなし）
row.to_csv(batch_X_csv_buffer, header=False, index=False)

# StringIOからCSV形式の文字列を取得
test_row = batch_X_csv_buffer.getvalue()

# 取得したCSV形式の文字列を表示
print(test_row)

# XGBoostモデルを使用してテストデータの行を予測するためにpredictメソッドを使用
xgb_predictor.predict(test_row)

## エンドポイント削除とバッチ変換

### エンドポイント削除
HTTPで予測を受け付けるサーバーが立ったまま = **課金が続く**  
→ 不要になったら必ず `delete_endpoint()` で削除する

### バッチ変換（一括予測）
エンドポイントを使った1件ずつの予測と違い、テストデータ全件を**まとめて予測**する方法

**メリット**：
- コスト効率が良い（予測後にインスタンスが自動終了）
- 大量データを一度に処理できる

**流れ**：
1. テストデータ（target列除き）をS3にアップロード
2. `transformer`オブジェクトを作成（変換設定）
3. `transform()`で一括予測実行 → 結果がS3に自動保存される
4. S3から結果をダウンロードして評価

In [ ]:
# XGBoost Predictorのエンドポイントとエンドポイント設定を削除するためにdelete_endpointメソッドを使用
xgb_predictor.delete_endpoint(delete_endpoint_config=True)

# テストデータから特徴量を抽出し、1列目以降を取得
batch_X = test.iloc[:,1:];

# バッチデータの入力ファイル名を指定
batch_X_file='batch-in.csv'

# バッチデータをS3にアップロード
upload_s3_csv(batch_X_file, 'batch-in', batch_X)

# バッチ変換の出力先を指定
batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)

# バッチ変換の入力データのパスを指定
batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

# XGBoostトランスフォーマーを作成
# instance_count: トランスフォーマーの実行に使用するインスタンスの数
# instance_type: トランスフォーマーのインスタンスのタイプ
# strategy: バッチ変換のストラテジーを指定（MultiRecordは複数の入力レコードに対応）
# assemble_with: バッチ変換の際に結果をどのようにまとめるか指定（Lineは行ごとにまとめる）
# output_path: バッチ変換の結果を保存するS3のパスを指定
xgb_transformer = xgb_model.transformer(instance_count=1,
                                       instance_type='ml.m4.xlarge',
                                       strategy='MultiRecord',
                                       assemble_with='Line',
                                       output_path=batch_output)

# バッチ変換を実行
# data: 変換対象のデータのパスを指定
# data_type: 変換対象のデータのタイプを指定（S3PrefixはS3上のプレフィックス内のデータを指定）
# content_type: 変換対象データのコンテンツタイプを指定
# split_type: 入力データをどのように分割するか指定（Lineは行ごとに分割）
xgb_transformer.transform(data=batch_input,
                         data_type='S3Prefix',
                         content_type='text/csv',
                         split_type='Line')

# バッチ変換が完了するのを待機
xgb_transformer.wait()

# バッチ変換完了が完了したことを表示
print('Batch transformation completed!')

In [ ]:
# boto3モジュールを使用してS3クライアントを作成
s3 = boto3.client('s3')

# S3バケットからオブジェクトを取得
obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))

# バッチ変換の結果として得られた予測結果を読み込む
# ここではCSVデータを読み込み、列名を指定してDataFrameに変換
target_predicted = pd.read_csv(io.BytesIO(obj['Body'].read()),sep=',',names=['target'])

# 閾値を設定して二値変換する関数を定義
# binary_convert関数では、確率が指定された閾値（ここでは0.65）より大きい場合は1、それ以外は0に変換されます。
def binary_convert(x):
    threshold = 0.65
    if x > threshold:
        return 1
    else:
        return 0

# 'class'列の各要素に二値変換を適用し、新しい 'binary' 列を作成
target_predicted_binary = target_predicted['target'].apply(binary_convert)

# 最初の10行を表示して、変換が正しく適用されていることを確認
print(target_predicted_binary.head(10))

# テストデータの最初の10行を表示
test.head(10)

# <font color="orange">4. モデルの評価指標</font>

## 混同行列（Confusion Matrix）

予測結果を4種類に分類してモデルの精度を評価する

|  | 予測:正常(Negative) | 予測:異常(Positive) |
|--|--|--|
| **実際:正常** | TN（真陰性）✅ | FP（偽陽性）❌ |
| **実際:異常** | FN（偽陰性）⚠️ | TP（真陽性）✅ |

## 主要な評価指標

| 指標 | 計算式 | 意味 |
|---|---|---|
| 感度(Sensitivity/TPR) | TP/(TP+FN) | 異常を正しく検出する率 |
| 特異度(Specificity/TNR) | TN/(TN+FP) | 正常を正しく判定する率 |
| 適合率(Precision) | TP/(TP+FP) | 陽性判定の正しさ |
| 偽陰性率(FNR) | FN/(TP+FN) | がんを見逃す率 |
| 正解率(Accuracy) | (TP+TN)/全件 | 全体の正解率 |
| AUC | ROC曲線の面積 | モデルの総合的な判別能力(0.5〜1.0) |

**医療シナリオでのポイント**：FNR（偽陰性率＝がんの見逃し率）が最重要。  
見逃しは命に関わるため、FNRを極力0に近づけることが優先。

In [ ]:
# テストデータのクラスラベル列を抽出する
test_labels = test.iloc[:,0]

# scikit-learnから混同行列を計算するためのメトリクスをインポート
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, auc

# テストデータの真のクラスラベルと予測されたクラスラベルに基づいて混同行列を計算
matrix = confusion_matrix(test_labels, target_predicted_binary)

# 混同行列をDataFrameに変換し、見やすくするために行と列にラベルを付ける
df_confusion = pd.DataFrame(matrix, index=['Nnormal','Abnormal'],columns=['Normal','Abnormal'])

# データの可視化に使用するSeabornモジュールとMatplotlibのpyplotモジュールをインポート
import seaborn as sns
import matplotlib.pyplot as plt

colormap = sns.color_palette("BrBG", 10)

# 混同行列（Confusion Matrix）をヒートマップとして可視化
sns.heatmap(df_confusion, annot=True, cbar=None, cmap=colormap)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.ylabel("True Class")
plt.xlabel("Predicted Class")
plt.show()

# 混同行列から真陰性（TN）、偽陽性（FP）、偽陰性（FN）、真陽性（TP）の各要素を計算
TN, FP, FN, TP = confusion_matrix(test_labels, target_predicted_binary).ravel()
print(f"True Negative (TN) : {TN}")
print(f"False Positive (FP): {FP}")
print(f"False Negative (FN): {FN}")
print(f"True Positive (TP) : {TP}")

# Sensitivity（感度）: 異常がある患者の異常を検出する確率
Sensitivity  = float(TP)/(TP+FN)*100
print(f"\nSensitivity or TPR: {Sensitivity}%")

# Specificity（特異度）: 正常である患者群に対する正常検出の確率
Specificity  = float(TN)/(TN+FP)*100
print(f"Specificity or TNR: {Specificity}%")

# Precision（適合率）: 陽性判定のうち実際に陽性の割合
Precision = float(TP)/(TP+FP)*100
print(f"Precision: {Precision}%")

# NPV（陰性予測値）: 陰性判定のうち実際に陰性の割合
NPV = float(TN)/(TN+FN)*100
print(f"Negative Predictive Value: {NPV}%")

# FPR（偽陽性率）
FPR = float(FP)/(FP+TN)*100
print(f"False Positive Rate: {FPR}%")

# FNR（偽陰性率）: 異常を見逃す率 ※医療では最重要
FNR = float(FN)/(TP+FN)*100
print(f"False Negative Rate: {FNR}%")

# FDR（誤検出率）
FDR = float(FP)/(TP+FP)*100
print(f"False Discovery Rate: {FDR}%")

# ACC（正解率）
ACC = float(TP+TN)/(TP+FP+FN+TN)*100
print(f"Accuracy: {ACC}%")

# AUC-ROC曲線
import numpy as np

print("\nValidation AUC", roc_auc_score(test_labels, target_predicted))

fpr, tpr, thresholds = roc_curve(test_labels, target_predicted)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % (roc_auc))
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")

ax2 = plt.gca().twinx()
ax2.plot(fpr, thresholds, markeredgecolor='r',linestyle='dashed', color='r')
ax2.set_ylabel('Threshold',color='r')
thresholds = [value for value in thresholds if np.isfinite(value)]
ax2.set_ylim([min(thresholds), max(thresholds)])
ax2.set_xlim([fpr[0],fpr[-1]])
print(plt.figure())

# <font color="orange">5. ハイパーパラメータチューニング</font>

SageMakerが**自動的に最適なハイパーパラメータの組み合わせ**を探索する機能

**チューニングするパラメータと範囲**：

| パラメータ | 範囲 | 意味 |
|---|---|---|
| `alpha` | 0〜100 | 正則化（大きいほど過学習防止） |
| `min_child_weight` | 1〜5 | 子ノードの最小重み |
| `subsample` | 0.5〜1 | 各ラウンドで使うデータの割合 |
| `eta` | 0.1〜0.3 | 学習率（小さいほど慎重に学習） |
| `num_round` | 1〜50 | 学習ラウンド数 |

**目的メトリクス**：`validation:error`（検証エラー率）を**最小化**

**注意**：`max_jobs=10` で10回試行。全範囲を探索すると数時間かかるため、実習では範囲を絞って使用。

In [ ]:
# SageMakerのハイパーパラメータ調整用のライブラリをインポート
from sagemaker.tuner import IntegerParameter, CategoricalParameter, ContinuousParameter, HyperparameterTuner

# XGBoost Estimatorの設定
xgb = sagemaker.estimator.Estimator(container,
                                    role=sagemaker.get_execution_role(), 
                                    instance_count=1,  # インスタンスの数を指定（1台）
                                    instance_type='ml.m4.xlarge',  # インスタンスのタイプを指定
                                    output_path='s3://{}/{}/output'.format(bucket, prefix),  # 出力先のS3パス
                                    sagemaker_session=sagemaker.Session())

# XGBoostのハイパーパラメータの設定
xgb.set_hyperparameters(eval_metric='error@.40',  # 評価メトリックを誤差率（error）としきい値0.40で設定
                        objective='binary:logistic',  # オブジェクティブ関数を二値分類のロジスティック回帰として設定
                        num_round=42  # ラウンド数を42回として設定
)

# ハイパーパラメータの探索範囲を指定
hyperparameter_ranges = {'alpha': ContinuousParameter(0, 100),  # 正則化項のハイパーパラメータalphaを0から100の連続値で指定
                        'min_child_weight': ContinuousParameter(1, 5),  # 子ノードの最小重みを1から5の連続値で指定
                        'subsample': ContinuousParameter(0.5, 1),  # サブサンプルの割合を0.5から1の連続値で指定
                        'eta': ContinuousParameter(0.1, 0.3),  # 学習率（eta）を0.1から0.3の連続値で指定
                        'num_round': IntegerParameter(1, 50)  # ラウンド数を1から50の整数値で指定
}

# ハイパーパラメータの調整のための目的のメトリクスの名前を設定します。ここでは検証データのエラーを最小化するようにします。
objective_metric_name = 'validation:error'

# 目的関数の種類を指定します。ここでは最小化が目標です。
objective_type = 'Minimize'

# ハイパーパラメータのチューニングを行うためのHyperparameterTunerオブジェクトを作成します。
tuner = HyperparameterTuner(xgb,
                            objective_metric_name,
                            hyperparameter_ranges,
                            max_jobs=10,  # 予算と利用可能な時間に応じて10以上に設定します。
                            max_parallel_jobs=1,  # 1つのトレーニングジョブのみを同時に実行します。
                            objective_type=objective_type,
                            early_stopping_type='Auto')  # 早期停止の種類を自動設定にします。

# ハイパーパラメータのチューニングを実行します。データチャネルとクラスメタデータは含めません。
tuner.fit(inputs=data_channels, include_cls_metadata=False)

# チューニングが完了するのを待ちます。
tuner.wait()

# チューニングが完了したことを表示
print('Tuning completed!')

# pprintモジュールからpprint関数をインポート
from pprint import pprint

# sagemaker.analyticsモジュールからHyperparameterTuningJobAnalyticsクラスをインポート
from sagemaker.analytics import HyperparameterTuningJobAnalytics

# 最新のハイパーパラメータチューニングジョブの名前を使用してHyperparameterTuningJobAnalyticsオブジェクトを作成
tuner_analytics = HyperparameterTuningJobAnalytics(tuner.latest_tuning_job.name, sagemaker_session=sagemaker.Session())

# ハイパーパラメータチューニングジョブの分析データをDataFrameとして取得
df_tuning_job_analytics = tuner_analytics.dataframe()

# ファイナルメトリクスの値でチューニングジョブの分析データをソート
df_tuning_job_analytics.sort_values(
    by=['FinalObjectiveValue'],
    inplace=True,
    ascending=False if tuner.objective_type == "Maximize" else True)

# 上位20モデルの詳細な分析データを表示
df_tuning_job_analytics.head(20)